# 04 - Identification workflow & conservation recommendations
This notebook runs the complete system from the architecture slide:

**iNaturalist image -> CNN -> predicted species -> IUCN Red List status -> recommendation**

* **Part A - Needs-ID triage.** Fetch recent iNaturalist bumble-bee observations that are still
  identified only to genus, suggest a species shortlist, cross-reference the Red List, and put
  potential threatened species at the top of the expert-review queue.
* **Part B - Where to invest.** A transparent rule-based system ranks geographic grid cells for
  conservation investment, and answers "where should we act to protect species *X*?".

In [ ]:
# --- Environment setup: works on Google Colab and locally -------------------------------
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/MaryCassieOdie/MSDS688-Case-Study.git"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    REPO_DIR = "/content/MSDS688-Case-Study"
    if not os.path.exists(REPO_DIR):
        # If the repo is private, upload the project folder to Drive instead and set
        # REPO_DIR = "/content/drive/MyDrive/MSDS688-Case-Study"
        subprocess.run(["git", "clone", "-q", REPO_URL, REPO_DIR], check=True)
    else:
        subprocess.run(["git", "-C", REPO_DIR, "pull", "-q"], check=False)
    # Data and outputs live on Drive so they survive Colab runtime resets.
    DRIVE_ROOT = "/content/drive/MyDrive/MSDS688-Case-Study"
    os.environ["PROJECT_DATA_DIR"] = f"{DRIVE_ROOT}/data"
    os.environ["PROJECT_OUTPUT_DIR"] = f"{DRIVE_ROOT}/outputs"
else:
    REPO_DIR = os.path.abspath("..")

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
# Optional: official IUCN Red List API token (otherwise iNaturalist's IUCN mirror is used)
# os.environ["IUCN_API_TOKEN"] = "..."   # on Colab, prefer: from google.colab import userdata

from pollinator_id import config
config.ensure_dirs()
print("data   ->", config.DATA_DIR)
print("output ->", config.OUTPUT_DIR)

In [ ]:
import json
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from pollinator_id import inaturalist, iucn, models, inference, gradcam
from pollinator_id import conservation_rules as cr
from pollinator_id.config import CLASS_NAMES

best = json.loads((config.OUTPUT_DIR / "best_model.json").read_text())["backbone"]
model = models.load_model(best)
iucn_table = pd.read_csv(config.IUCN_CSV)
redlist = iucn.RedListReference(iucn_table)
occ = pd.read_csv(config.OCCURRENCES_CSV)
client = inaturalist.INatClient()
print("model:", best)

## Part A - Triage of unidentified observations
**Important caveat:** the classifier is *closed-set*: it can only answer with one of the 12 species
it was trained on, while North America has ~50 bumble-bee species. Low-confidence suggestions are
labelled *uncertain*, and every output is a **recommendation for expert review**, never an
automatic identification.

In [ ]:
needs = inference.fetch_needs_id_observations(client, "Bombus", n=60)
queue = inference.triage(model, best, needs, redlist, review_threshold=0.5)
cols = ["url", "place_guess", "observed_on", "pred_label", "confidence", "pred_iucn_code", "review_note", "alternatives"]
queue[cols].head(20).style.format({"confidence": "{:.0%}"})

In [ ]:
show = queue.head(8).reset_index(drop=True)
fig, axes = plt.subplots(2, len(show), figsize=(2.7 * len(show), 7), gridspec_kw={"hspace": 0.35})
for j, r in show.iterrows():
    ov, *_ = gradcam.explain(model, best, r.image)
    axes[0, j].imshow(r.image.astype("uint8"))
    axes[0, j].set_title(f"obs {r.observation_id}", fontsize=8)
    axes[1, j].imshow(ov)
    axes[1, j].set_title(f"{r.pred_label.split()[1]} [{r.pred_iucn_code}]\n{r.confidence:.0%} - {r.review_note}",
                         fontsize=8, color="tab:red" if r.pred_threatened else "black")
for a in axes.ravel(): a.axis("off")
fig.suptitle("Top of the expert-review queue (red = suggested threatened species)")
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / "triage_queue.png", dpi=130)

In [ ]:
summary = queue.groupby(["pred_label", "pred_iucn_code"]).agg(n=("observation_id", "size"),
                                                               mean_conf=("confidence", "mean"))
print(f"{queue.pred_threatened.sum()} of {len(queue)} unidentified observations suggested as threatened species")
summary.sort_values("n", ascending=False).round(2)

## Part B - Rule-based conservation recommendations
Occurrence records are binned into a 1 deg x 1 deg grid. Each rule that fires in a cell adds
priority points and a plain-language recommendation:

| Rule | Points | Fires when |
|---|---|---|
| Critical refuge | 5 | CR/EN species with >= 3 records in the last 5 years |
| Vulnerable stronghold | 3 | VU species with >= 3 recent records |
| Threatened co-occurrence hotspot | 3 | >= 2 threatened species recently present |
| Possible local decline | 2 | threatened species with >= 3 older records but none recent |
| Under-surveyed area | 1 | threatened species present but < 20 records of any study species |
| Pending expert review | 1 | model flagged an unverified threatened sighting (>= 70% confidence) |

Tiers: **High** >= 8, **Medium** 4-7, **Low** 1-3. Thresholds live in `RuleSettings` and every
ranking lists the rules that produced it, so planners can audit or adjust the logic.

In [ ]:
settings = cr.RuleSettings(cell_deg=1.0, recent_years=5)
pending = queue[["latitude", "longitude", "pred_label", "pred_iucn_code", "confidence"]].dropna()
cells = cr.evaluate(occ, iucn_table, settings, pending=pending)
cells.to_csv(config.OUTPUT_DIR / "conservation_priorities.csv", index=False)
print(cells.priority_tier.value_counts().to_dict())
cells.head(15)[["cell_id", "center_lat", "center_lon", "priority_score", "priority_tier",
                "threatened_species", "rules_fired"]]

In [ ]:
ax = cr.plot_priority_map(cells, occ, title="Conservation priority by 1 deg grid cell")
ax.set_xlim(-170, -50); ax.set_ylim(15, 70)
plt.savefig(config.FIGURE_DIR / "priority_map.png", dpi=150, bbox_inches="tight")

In [ ]:
# Interactive map (click a cell for the rules that fired). folium is preinstalled on Colab.
try:
    fmap = cr.folium_map(cells, settings.cell_deg)
    fmap.save(str(config.FIGURE_DIR / "priority_map.html"))
    display(fmap)
except ImportError:
    print("pip install folium for the interactive map")

### Where should we invest to protect a given species?

In [ ]:
TARGET = "Bombus affinis"   # try any threatened species in scope
rec = cr.recommend_for_species(TARGET, occ, iucn_table, settings, top_n=10)
print(f"Top locations for {TARGET} [{redlist.status(TARGET)['iucn_code']}]")
rec[["cell_id", "center_lat", "center_lon", "priority_tier", "species_recent", "species_total",
     "species_last_year", "rules_fired"]]

In [ ]:
for _, r in rec.head(3).iterrows():
    print(f"\n## Cell {r.cell_id} ({r.center_lat:.1f}, {r.center_lon:.1f}) - {r.priority_tier} priority")
    for line in r.recommendations.split(" | "):
        print("  -", line)

### Sensitivity analysis
Rules encode judgement calls. How stable are the rankings under different grid sizes and
definitions of "recent"? Rankings that persist across settings are the most defensible investments.

In [ ]:
rows = []
for deg in (0.5, 1.0, 2.0):
    for yrs in (3, 5, 8):
        c = cr.evaluate(occ, iucn_table, cr.RuleSettings(cell_deg=deg, recent_years=yrs))
        rows.append({"cell_deg": deg, "recent_years": yrs, "cells_scored": len(c),
                     "high": int((c.priority_tier == "High").sum()),
                     "decline_flags": int(c.rules_fired.str.contains("decline").sum())})
pd.DataFrame(rows).pivot_table(index="cell_deg", columns="recent_years", values=["high", "decline_flags"])

## Limitations & responsible use
* **Closed-set classifier** - species outside the 12 classes will be forced into one of them;
  confidence thresholds and expert review are essential. Extending to all regional *Bombus*
  species is the natural next step.
* **Sampling bias** - citizen-science records follow people, roads and cities. "No recent
  records" can mean "nobody looked", which is why the under-surveyed rule recommends monitoring
  before investment.
* **Location sensitivity** - precise locations of threatened species can enable collection or
  disturbance. Outputs here use 1 deg cells and respect iNaturalist's coordinate obscuring;
  do not publish finer-grained threatened-species locations.
* **Status data** - IUCN categories are global; regional/national listings (e.g. US ESA,
  Canada SARA) may differ and could be added as additional rules.
* **Licensing** - photos are CC0/CC-BY/CC-BY-NC; credits are in `data/ATTRIBUTION.csv`. Use is
  non-commercial and educational.